### Delta Table Cloning

- You can create a copy of an existing Delta Lake table on Databricks at a specific version using the **`clone`** command. 
- Clones can be either **deep** or **shallow**.

- A **deep clone** is a clone that **copies the source table data** to the clone target in addition to the metadata of the existing table.

- A **shallow clone** is a clone that **does not copy the data files** to the clone target. The table metadata is equivalent to the source. These clones are cheaper to create.

- Any changes made to either deep or shallow clones affect only the clones themselves and not the source table.

- A cloned table has an independent history from its source table. Time travel queries on a cloned table do not work with the same inputs as they work on its source table.


In [0]:
CREATE SCHEMA IF NOT EXISTS demodb;
USE demodb;

In [0]:
SELECT current_schema()

In [0]:
CREATE TABLE IF NOT EXISTS users_managed 
(id INT, name STRING, age INT, gender STRING);

In [0]:
INSERT INTO users_managed 
VALUES 
(1, "Raju", 48, "Male"), (2, "Ramesh", 25, "Male"),
(3, "Ramya", 38, "Female"), (4, "Radhe", 45, "Female"),
(5, "Ravi", 28, "Male"), (6, "Raheem", 25, "Male"),
(7, "Revati", 40, "Female"), (8, "Raghu", 35, "Male");

In [0]:
DESCRIBE EXTENDED users_managed

In [0]:
SELECT * FROM users_managed

In [0]:
UPDATE users_managed 
SET age = age + 1
WHERE gender = 'Male'

In [0]:
DELETE FROM  users_managed WHERE id = 8

In [0]:
SELECT * FROM users_managed

In [0]:
DESCRIBE HISTORY users_managed

#### Deep Clone

- Deep clones do not depend on the source from which they were cloned, but are expensive to create because a deep clone copies the data as well as the metadata.

#### Shallow Clone

- Shallow clones reference data files in the source directory. If you run vacuum on the source table, clients can no longer read the referenced data files and a FileNotFoundException is thrown. 

In [0]:
%python

from delta.tables import *
dt_users_managed = DeltaTable.forName(spark, "users_managed")

In [0]:
%python
dt_users_managed.toDF().display()

In [0]:
SHOW TABLES

In [0]:
%python

# clone the source at latest version
dt_users_managed.clone (
  target="users_cloned", 
  isShallow=False, 
  replace=False
) 

In [0]:
SHOW TABLES

In [0]:
SELECT * FROM users_cloned

In [0]:
DESCRIBE EXTENDED users_cloned

In [0]:
DESCRIBE HISTORY users_cloned

In [0]:
%python
dt_users_cloned = DeltaTable.forName(spark, "users_cloned")

In [0]:
%python
dt_users_cloned.update(
  condition = "gender = 'Male'",
  set = {"age": "age + 1"}
)

In [0]:
%python
### DESC HISTORY users_cloned
dt_users_cloned.history().display()

In [0]:
%python
spark.table("demodb.users_cloned").display()

In [0]:
%python

dt_users_managed.clone (
  target="users_cloned_shallow", 
  isShallow=True, 
  replace=True
) 

In [0]:
SHOW TABLES

In [0]:
SELECT * FROM users_cloned_shallow

In [0]:
DESC EXTENDED users_cloned_shallow

In [0]:
DESCRIBE HISTORY users_cloned_shallow

In [0]:
%python
dt_users_managed.history().display()

In [0]:
%python

# clone the source at a specific version
dt_users_managed.cloneAtVersion(
    version=1, 
    target="users_cloned_shallow_v1", 
    isShallow=True, 
    replace=False
) 

In [0]:
%python

# clone the source at a specific version
dt_users_managed.cloneAtTimestamp(
    timestamp="2026-03-18 06:51:05", 
    target="users_cloned_ts", 
    isShallow=False, 
    replace=False
) 

In [0]:
SHOW TABLES

In [0]:
DESCRIBE EXTENDED users_cloned_ts

#### Cleanup

In [0]:
DROP SCHEMA demodb CASCADE